# Taller de la semana 4 — SOLUCIÓN DE REFERENCIA (solo para el docente)

**Enunciado:** [`README.md`](README.md) · **Plan de clase:** [`../../weeks/week-04.md`](../../weeks/week-04.md) · **Motor proporcionado:** [`gridworld.py`](gridworld.py) · **Ayudantes del duelo:** [`duel.py`](duel.py)

Este notebook presenta la **explicación didáctica** del taller de la semana 4.
**Importa el código de referencia de [`solution.py`](solution.py)** sin volver a copiarlo,
de modo que la solución mostrada es exactamente la que evalúan las pruebas proporcionadas.
La garantía de corrección se comprueba por separado: el comando documentado
`python3 studios/_verify_solutions.py week-04` ejecuta `test_heuristics.py` sin modificaciones
contra `solution.py`.

> No distribuir. Se excluye de la versión para estudiantes mediante `studios/.gitignore`.


In [1]:
# --- Preparación: añade la carpeta de la semana (solution/gridworld/duel) y la
# raíz del repositorio (aicourse) a la ruta de importación desde las ubicaciones previstas. ---
import sys, pathlib
# POO: pathlib.Path es una clase; cwd devuelve un objeto que representa el directorio actual.
# El operador / combina rutas y exists() consulta si el archivo existe.
here = pathlib.Path.cwd()
week = here if (here / "solution.py").exists() else here / "studios" / "week-04"
root = week.parent.parent
for p in (str(week), str(root)):
    if p not in sys.path:
        # Método de list: da prioridad a estas rutas al resolver los módulos importados.
        sys.path.insert(0, p)

import inspect
import solution
from gridworld import GridProblem, astar, load_instances, render, MIN_COST, MAX_COST

# Diccionario de tamaños a listas de cuadrículas; se lee el JSON sin modificarlo.
BANK = load_instances()
print(f"MIN_COST={MIN_COST} MAX_COST={MAX_COST}; sizes:", sorted(BANK))

MIN_COST=1 MAX_COST=8; sizes: [5, 8, 12, 16]


## Tarea 1 — Cuatro heurísticas: tres admisibles y una deliberadamente incorrecta

`_optimal_cost` de las pruebas usa UCS, es decir, A* con `h=0`, como referencia.
Es óptima porque todos los costos por paso son > 0. Cada afirmación de abajo se compara con ella.


In [2]:
# Las funciones son objetos: fn referencia cada heurística e inspect muestra su código real.
for fn in (solution.h_zero, solution.h_manhattan, solution.h_manhattan_scaled,
           solution.h_euclidean_scaled, solution.h_bad):
    print(inspect.getsource(fn))

def optimal_cost(grid):
    # Instancia el problema y pasa una función heurística constante cero para obtener el óptimo de referencia.
    node, _ = astar(GridProblem(grid), lambda p, st: 0)
    return node.g

def h_zero(problem: GridProblem, state) -> int:
    """Admisible, pero no informa: A* se convierte en búsqueda de costo uniforme."""
    return 0

def h_manhattan(problem: GridProblem, state) -> int:
    """Distancia de Manhattan sin escalar hasta el objetivo.

    Aquí es admisible porque ``MIN_COST == 1``: cada paso cuesta al menos 1,
    así que |dr|+|dc| no supera el costo restante real. Si ``MIN_COST`` fuera 2,
    seguiría subestimando y sería admisible, pero la versión escalada de abajo
    la dominaría: proporcionaría una cota inferior más ajustada.
    """
    # Acceso a un atributo del objeto problema; desempaqueta la fila y columna del objetivo.
    gr, gc = problem.goal
    # state es una tupla de coordenadas del nodo evaluado, distinta de la instancia problem.
    r, c = state
    # abs es una función incorporada: suma distancias por fila y columna, sin diagonales.
    return abs(r - gr) + abs(c - gc)

def h_manhattan_scaled(problem: GridProblem, state) -> int:
    """Dist

### Las garantías que deben CUMPLIRSE

- `test_h_zero_admissible_and_optimal` — `h_zero` es admisible por definición ⇒ A* devuelve el óptimo real.
- `test_manhattan_scaled_is_admissible` — `Manhattan × MIN_COST` nunca sobreestima el costo restante real.
- `test_manhattan_scaled_dominates_zero` — una heurística que domina a `h_zero` **no expande más** nodos.
- `test_manhattan_scaled_returns_optimal_cost` — además, sigue devolviendo el óptimo.


In [3]:
# h_zero óptima en las 40 cuadrículas
for size in BANK:
    for grid in BANK[size]:
        node, _ = astar(GridProblem(grid), solution.h_zero)
        assert node.g == optimal_cost(grid)
print("h_zero admissible => optimal on all 40 grids")

# Manhattan escalada: admisible en la muestra, domina a cero y conserva la optimalidad
for size in BANK:
    for grid in BANK[size]:
        prob = GridProblem(grid)
        for state in [prob.initial, prob.goal, (prob.rows // 2, prob.cols // 2)]:
            # sub y prob son instancias distintas: cambiar sub.initial no altera prob.initial.
            # Así medimos el costo restante desde otro estado manteniendo el mismo objetivo.
            sub = GridProblem(grid); sub.initial = state
            rem, _ = astar(sub, lambda p, st: 0)
            # .g es el costo del Node objetivo; la tolerancia contempla diferencias numéricas mínimas.
            assert solution.h_manhattan_scaled(prob, state) <= rem.g + 1e-9
        _, exp_zero = astar(prob, solution.h_zero)
        _, exp_manh = astar(prob, solution.h_manhattan_scaled)
        # Compara expansiones para el mismo problema, sin depender de la velocidad del equipo.
        assert exp_manh <= exp_zero
        node, _ = astar(prob, solution.h_manhattan_scaled)
        assert node.g == optimal_cost(grid)
print("h_manhattan_scaled: admissible, dominates h_zero, and stays optimal on all 40 grids")

h_zero admissible => optimal on all 40 grids
h_manhattan_scaled: admissible, dominates h_zero, and stays optimal on all 40 grids


## La garantía falla: una heurística no admisible rompe la optimalidad

Este es **el objetivo de la semana** (`test_inadmissible_heuristic_returns_suboptimal_path`).
`h_bad = MAX_COST × Manhattan` supone que cada paso restante cuesta 8 y sobreestima cuando
el camino óptimo pasa por celdas `.` baratas. A* es óptima *porque* se cumple la admisibilidad;
`h_bad` permite observar el fallo de esa propiedad: A* devuelve un camino estrictamente
más caro que el óptimo real en al menos una instancia.


In [4]:
broken = 0; total = 0; example = None
for size in BANK:
    for grid in BANK[size]:
        node, _ = astar(GridProblem(grid), solution.h_bad)
        opt = optimal_cost(grid)
        # Solo cuenta un fallo si el costo obtenido es estrictamente mayor que el óptimo recalculado.
        if node.g > opt:
            broken += 1
            if example is None:
                example = (size, node.g, opt)
        total += 1
# Esta aserción PASA cuando observa al menos un fallo: es el resultado deliberado del ejercicio.
assert broken > 0
# Desempaqueta la primera instancia con fallo para mostrar un ejemplo concreto.
sz, got, opt = example
print(f"GUARANTEE FAIL confirmed: h_bad broke optimality on {broken}/{total} instances.")
print(f"  e.g. a {sz}x{sz} grid: A*+h_bad returned cost {got}, true optimum is {opt}.")
# En contraste, A* con la heurística admisible conserva el óptimo en el MISMO banco:
assert all(astar(GridProblem(g), solution.h_manhattan_scaled)[0].g == optimal_cost(g)
           for size in BANK for g in BANK[size])
print("  (h_manhattan_scaled stays optimal on every one of them — admissibility is the difference.)")

GUARANTEE FAIL confirmed: h_bad broke optimality on 22/40 instances.
  e.g. a 5x5 grid: A*+h_bad returned cost 20, true optimum is 8.
  (h_manhattan_scaled stays optimal on every one of them — admissibility is the difference.)


## Tarea 2 — El duelo: LLM frente a A*, demostración sin interacción

`duel.py` proporciona el prompt exacto, un verificador y cuatro categorías:
`illegal`, `suboptimal`, `wrong_cost` y `correct`. La regla fundamental:
**nunca confíes en el costo declarado por el modelo; vuelve a calcularlo con el verificador**.
Por eso `wrong_cost` tiene su propia categoría.

Usamos el backend determinista `echo`, sin interacción, para que el notebook
pueda ejecutarse en cualquier entorno.

> **`echo` es un SIMULADOR determinista, no un modelo.** Devuelve una cadena fija:
> `parse_reply` no encuentra `PATH:` ni `COST:` y todas las respuestas se clasifican como `illegal`.
> Esto solo comprueba el *proceso de evaluación*, no un modelo. Para obtener una curva real
> con A* estable en el 100 % y el LLM descendiendo, ejecuta de nuevo con `backend="ollama"`
> o `backend="manual"`. Nunca reportes la salida de `echo` como resultado de un modelo.


In [ ]:
import os
from aicourse.llm import LLM
from duel import prompt_for, parse_reply, score_llm_reply

backend = os.environ.get("AICOURSE_NB_BACKEND", "echo")   # echo = simulador sin interacción
# POO: LLM es la clase y llm una instancia configurada con este backend.
# complete() es su método para obtener respuestas; .text accede al atributo textual del resultado.
llm = LLM(backend=backend)

grid = BANK[5][0]
print("Sample 5x5 grid:")
print(render(grid))
print("Ground-truth optimum via A*:", optimal_cost(grid), "\n")

# Comprueba el proceso con unas instancias: echo produce 'illegal', como se explicó arriba.
from collections import Counter
# Counter es una clase de diccionario especializada en conteos; las categorías empiezan en cero.
cats = Counter()
for g in BANK[5][:3]:
    reply = llm.complete(prompt_for(g)).text
    # Se entrega el texto de respuesta al analizador y se desempaquetan ruta y costo declarado.
    path, cost = parse_reply(reply)
    # score_llm_reply devuelve una instancia Scored; .category consulta uno de sus atributos.
    cats[score_llm_reply(g, path, cost).category] += 1
print("backend:", backend, "(echo = FAKE)" if backend == "echo" else "")
print("category counts (echo pipeline):", dict(cats))

# Comprueba que el verificador detecta un costo declarado falso: entrega el PATH óptimo real
# con un costo declarado INCORRECTO: debe clasificarse como 'wrong_cost', no 'correct'.
node, _ = astar(GridProblem(grid), solution.h_manhattan_scaled)
# node.path() es un método de Node; join() es un método de str que une las letras sin separadores.
optimal_path = "".join(node.path())
# Conserva el camino óptimo y altera solo el dato declarado para aislar el fallo de costo.
scored = score_llm_reply(grid, optimal_path, node.g + 999)
print(f"\noptimal path {optimal_path!r} with a LIED cost -> category={scored.category!r} "
      f"(checker_cost={scored.checker_cost}, optimal={scored.optimal_cost})")
assert scored.category == "wrong_cost"
print("The checker recomputed the cost and refused the model's self-report (never trust it).")